<a href="https://colab.research.google.com/github/vpandey6398/rock-paper-scissor/blob/master/customer_churn_predictiom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
df=pd.read_csv("/Telco_Customer_Churn_Expanded.csv")

In [ ]:
df.head()

In [ ]:
df.columns.tolist()

In [ ]:
df.set_index('customerID', inplace=True)

In [ ]:
df.shape

In [ ]:
df.info()

Data Cleaning

In [ ]:
# Replace empty strings with NaN
df['TotalCharges'] = df['TotalCharges'].replace(' ', pd.NA)

In [ ]:
#convert to numeric
df['TotalCharges']=pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(subset='TotalCharges', axis=0, inplace=True)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
#check duplicate entreies
df.index.duplicated().sum()

In [ ]:
#Encode churn: No=0,Yes=1
df['Churn']=df['Churn'].str.strip().map({'No':0 , 'Yes':1})

In [ ]:
df.head()

Exploratory Data Analysis

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

In [ ]:
#check class distribution
df['Churn'].value_counts(normalize=True)*100

Visualize Categorical Features vs Churn

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

#set the figure size
plt.figure(figsize=(12,16))

useful_cat_features=[
    'Contract', 'InternetService', 'PaymentMethod', 'TechSupport', 'OnlineSecurity', 'PaperlessBilling'
]
for i, col in enumerate(useful_cat_features, 1):
  plt.subplot(len(useful_cat_features), 1, i)
  sns.countplot(data=df, x=col, hue='Churn')
  plt.title(f'Churn Count by {col}')
  plt.xticks(rotation=45)
  plt.tight_layout()

  plt.show

Visualize Numeric features vs Churn

In [ ]:
num_features = ['TotalCharges', 'MonthlyCharges', 'tennure']
plt.figure(figsize=(12, 16))

for i, col in enumerate(num_features, 1):
  plt.subplot(len(num_features), 1, i)
  sns.boxplot(data=df, x='Churn', y=col)
  plt.title(f'Churn vs {col}')
  plt.xticks(rotation=45)
  plt.tight_layout()

  plt.show

Correlation Heatmap (for numeric features)

In [ ]:
#select only numeric features
df_numeric = df.select_dtypes(include=['number'])
#compute correlation matrix
corr = df_numeric.corr()
#plot heatmap
plt.figure(figsize=(12,10))
sns.heatmap(corr,annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

Data Preprocessing

**Encoding Categorical Variables**

In [ ]:
#get only object type columns
cat_cols=df.select_dtypes(include=['object']).columns
cat_cols

In [ ]:
#seperate binary and multi-class features
binary_features =[col for col in cat_cols if df[col].nunique() == 2]
multi_class_features = [col for col in cat_cols if df[col].nunique()>2]


In [ ]:
binary_features

In [ ]:
multi_class_features

In [ ]:
# if you want a more format way
summary = pd.Dataframe({
    'Feature': cat_cols,
    'Unique Values': [df[col].nunique() for col in cat_cols],
    "Type" : ["Binary" if df[col].nunique() == 2 else "Multi-class" for col in cat_cols]
})
summary.sort_values(by='Unique Values')

**Encoding categorical features**

In [ ]:
from sklearn.preprocessing import LabelEncoder

#label encoding for binary class
le = LabelEncoder()
for col in binary_features:
  df[col] = le.fit_transform(df[col])

In [ ]:
df[binary_features].head()

one hot encoding for multi class

In [ ]:
df = pd.get_dummies(df, columns=multi_class_features, drop_first=True)

In [ ]:
df.head()

In [ ]:
# Identify bool columns
bool_cols = df.select_dtypes(include=['bool']).columns

# Convert bool columns to int
df[bool_cols] = df[bool_cols].astype(int)

In [ ]:
df.head()

**Feature Scalling**


*   scaling numerical features,like tenure,MonthlyCharges, and TotalCharges, to ensure consistent value ranges



In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

numeric_cols = ['Tenure', 'TotalCharges', 'MonthlyCharges']

df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

In [ ]:
df[numeric_cols]

**Seprate Features and Target Variable**
* X: the input features(independent variables)
* y: the target variable(what we want to predict)

In [ ]:
# Target variable
y = df['Churn']

x=df.drop('churn', axis=1)

**Train Test Split**
* Training set: Used to Train the model.
* Testing set: Used to assess performance on unseen data

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

**Handle Class Imbalance in Trainig Data**

since our dataset is imbalanced-with around 73"% 'No' and 27% 'Yes' for churn - it's important to balance the classes in the training set.This ensures the model does not become biased toward the majority class

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
x_train_resampled, y_train_resampled = smote.fit_resample(x_train, y_train)

In [ ]:
y_train_resampled.value_counts()

**Build a Model Pipeline**

A pipeline allows you to:

* organizing preprocessing and model training into a single workflow

* Ensure consistent application of transformations

* prevent data leakage




In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report


**Tranin and evaluate the models**

In [ ]:
# Models to compare
models = {
    'Logistic Regression': LogisticRegression(solver='liblinear', random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42)
}

In [ ]:
from sklearn.metrics import roc_auc_score

#pipelines
for name, model in models.items():
  pipe = Pipeline([
      ('scaler', StandardScaler()),
      ('model', model)
  ])
  # Fit the pipeline
  pipe.fit(x_train_resampled, y_train_resampled)

  y_pred = pipe.predict(x_test)
  y_pred_proba = pipe.predict_proba(x_test)[:, 1] # Get probability of positive class

  print(f"Model: {name}")

  report = classification_report(y_test, y_pred)
  print("Classification Report:")
  print(report)

  auc = roc_auc_score(y_test, y_pred_proba)
  print(f"AUC: {auc:.4f}")
  print("-" * 50)